# Day 09. Exercise 02
# Metrics

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import joblib

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df=pd.read_csv("../data/day-of-week-not-scaled.csv")
previous_df=pd.read_csv("../data/dayofweek.csv")
df['dayofweek']=previous_df['dayofweek']

X=df.drop('dayofweek', axis=1)
y=df['dayofweek']

X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.
2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [3]:
svc = SVC(random_state=21,kernel='rbf',gamma='auto',class_weight=None,C=10,probability=True)
svc.fit(X_train, y_train)
pred = svc.predict(X_test)
pred_proba = svc.predict_proba(X_test)

In [4]:
accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, average='weighted')
recall = recall_score(y_test, pred, average='weighted')
roc_auc = roc_auc_score(y_test, pred_proba, multi_class='ovr')

print(f'accuracy is {accuracy:.5}')
print(f'precision is {precision:.5}')
print(f'recall is: {recall:.5}')
print(f'roc_auc is {roc_auc:.5}')

accuracy is 0.88757
precision is 0.89267
recall is: 0.88757
roc_auc is 0.97751


## 3. Decision tree

1. The same task for decision tree

In [5]:
dt = DecisionTreeClassifier(random_state=21,class_weight='balanced',criterion='gini',max_depth=21)
dt.fit(X_train, y_train)
pred = dt.predict(X_test)
pred_proba = dt.predict_proba(X_test)

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, average='weighted')
recall = recall_score(y_test, pred, average='weighted')
roc_auc = roc_auc_score(y_test, pred_proba, multi_class='ovr')

print(f'accuracy is {accuracy:.5}')
print(f'precision is {precision:.5}')
print(f'recall is: {recall:.5}')
print(f'roc_auc is {roc_auc:.5}')

accuracy is 0.88462
precision is 0.88765
recall is: 0.88462
roc_auc is 0.93459


## 4. Random forest

1. The same task for random forest.

In [6]:
rf = RandomForestClassifier(random_state=21, n_estimators=100, max_depth=24, class_weight='balanced', criterion='entropy')
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
pred_proba = rf.predict_proba(X_test)

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, average='weighted')
recall = recall_score(y_test, pred, average='weighted')
roc_auc = roc_auc_score(y_test, pred_proba, multi_class='ovr')

print(f'accuracy is {accuracy:.5}')
print(f'precision is {precision:.5}')
print(f'recall is: {recall:.5}')
print(f'roc_auc is {roc_auc:.5}')

accuracy is 0.92604
precision is 0.92754
recall is: 0.92604
roc_auc is 0.98834


## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [7]:
rf = RandomForestClassifier(random_state=21, n_estimators=100, max_depth=24, class_weight='balanced', criterion='entropy')
rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=24, random_state=21)

In [8]:
df_analysis = df.copy()
df_analysis['preds'] = rf.predict(X)

df_analysis['is_error'] = (df_analysis['dayofweek'] != df_analysis['preds']).astype(int)

def get_error_percentage(group_col_prefix, original_df):
    results = {}
    
    cols = [c for c in original_df.columns if c.startswith(group_col_prefix)]
    
    if group_col_prefix == 'dayofweek':
        grouped = original_df.groupby('dayofweek')['is_error'].mean() * 100
        return grouped.sort_values(ascending=False)

    for col in cols:
        subset = original_df[original_df[col] == 1.0]
        if len(subset) > 0:
            error_perc = subset['is_error'].mean() * 100
            results[col] = error_perc
            
    return pd.Series(results).sort_values(ascending=False)

print(get_error_percentage('dayofweek', df_analysis))

print("\nlabname")
print(get_error_percentage('labname', df_analysis).head())

print("\nusers")
print(get_error_percentage('uid_user', df_analysis).head())

dayofweek
0    4.411765
4    2.884615
5    1.845018
1    1.459854
2    1.342282
3    0.757576
6    0.561798
Name: is_error, dtype: float64

labname
labname_lab03      100.000000
labname_laba04       3.370787
labname_laba06s      3.278689
labname_lab05s       2.777778
labname_laba06       2.083333
dtype: float64

users
uid_user_6     16.666667
uid_user_22    14.285714
uid_user_27     4.347826
uid_user_16     3.125000
uid_user_18     2.857143
dtype: float64


In [9]:
joblib.dump(rf, 'best_random_forest_model.joblib')

['best_random_forest_model.joblib']

## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

In [10]:
def count_metrics(models, params):
    results = {}
    
    for model, model_params in zip(models, params):
        curr_model = model(**model_params, random_state=21)
        
        curr_model.fit(X_train, y_train)
        pred = curr_model.predict(X_test)
        pred_proba = curr_model.predict_proba(X_test)

        metrics = {
            'accuracy': accuracy_score(y_test, pred),
            'precision': precision_score(y_test, pred, average='weighted'),
            'recall': recall_score(y_test, pred, average='weighted'),
            'roc_auc': roc_auc_score(y_test, pred_proba, multi_class='ovr')
        }
        
        results[model.__name__] = metrics
    
    return results

In [11]:
models = [RandomForestClassifier, SVC]
params = [
    {'n_estimators': 100, 'max_depth': 31},
    {'kernel': 'linear', 'gamma': 'auto', 'probability': True}
]
metrics_results = count_metrics(models, params)
metrics_results

{'RandomForestClassifier': {'accuracy': 0.9378698224852071,
  'precision': 0.9390993534287375,
  'recall': 0.9378698224852071,
  'roc_auc': 0.9877207268160056},
 'SVC': {'accuracy': 0.7189349112426036,
  'precision': 0.7271369623885529,
  'recall': 0.7189349112426036,
  'roc_auc': 0.9114244836329188}}